In [ ]:
#Compruebo que tengo la A100
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive, userdata
import os

drive.mount("/content/drive")

#La llave de HF desde los secretos de Colab (cambiar el nombre si el secreto se llama distinto)
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("HF_TOKEN configurado:", bool(os.environ["HF_TOKEN"]))

In [ ]:
%cd /content
# vLLM, la misma versión que en RunPod
!pip install -q uv
!uv venv vllm
!uv pip install --python vllm/bin/python "vllm==0.33.0"

#FlashInfer kernels are disabled fix
!./vllm/bin/vllm download-kernels

In [ ]:
from huggingface_hub import HfApi

#Lista de subidas al repo, de la más nueva a la más antigua
#Cada subida de un epoch es un commit: copio el hash del que quiera evaluar
for commit in HfApi().list_repo_commits("gabmfx1/salamandra-7b-eagle3-alia"):
    print(commit.created_at.strftime("%H:%M"), commit.commit_id, commit.title)

In [ ]:
#Qué checkpoint corresponde a cada commit (la subida más antigua es el epoch 0, la siguiente el epoch 1)
CHECKPOINTS = {
    "eagle3_ep0": "PEGAR_AQUI_HASH_DEL_EPOCH_0",
    "eagle3_ep1": "PEGAR_AQUI_HASH_DEL_EPOCH_1",
}

#Bajo solo lo necesario para servir, sin el estado del optimizador
for etiqueta, revision in CHECKPOINTS.items():
    !./vllm/bin/hf download gabmfx1/salamandra-7b-eagle3-alia --revision {revision} --local-dir /content/{etiqueta} --exclude "optimizer_state_dict.pt" "scheduler_state_dict.pt" "train.log"

#Y el modelo base, para no esperar luego
!./vllm/bin/hf download BSC-LT/salamandra-7b-instruct-2606

In [ ]:
import json

#Miro qué dice el checkpoint: modelo grande que usa y cuántos tokens propone el draft por paso
with open("/content/eagle3_ep0/config.json") as archivo:
    config = json.load(archivo)

print(json.dumps(config.get("speculators_config", config), indent=2)[:1500])

In [ ]:
#Métricas de validación de cada epoch (deberían mejorar del 0 al 1)
for etiqueta in CHECKPOINTS:
    print(etiqueta)
    !cat /content/{etiqueta}/val_metrics.json
    print()

In [ ]:
from collections import Counter

#Elijo cuántas prompts por idioma y cuánto puede generar cada respuesta
N_POR_IDIOMA = 20
MAX_TOKENS = 512

def prompt_de(conversacion):
    #Me quedo con los mensajes hasta la primera respuesta del asistente
    mensajes = []
    for mensaje in conversacion["messages"]:
        if mensaje["role"] == "assistant":
            break
        mensajes.append({"role": mensaje["role"], "content": mensaje["content"]})
    return mensajes

prompts = {"es": [], "ca": [], "eu": [], "gl": []}

with open("/content/drive/MyDrive/salamandra-eagle3/alia_eval_prompts.jsonl", encoding="utf-8") as archivo:
    for linea in archivo:
        conversacion = json.loads(linea)
        idioma = conversacion["lang"]
        if idioma in prompts and len(prompts[idioma]) < N_POR_IDIOMA:
            prompts[idioma].append(prompt_de(conversacion))

print({idioma: len(lista) for idioma, lista in prompts.items()})

In [ ]:
import subprocess, signal, time, re
import requests

URL = "http://127.0.0.1:8000"

def arrancar(modelo, log):
    #Mismo servidor para las dos versiones, sin prefix caching para que no influya en la medida
    with open(log, "w") as archivo:
        servidor = subprocess.Popen(
            [
                "./vllm/bin/vllm", "serve", modelo,
                "--max-model-len", "8192",
                "--port", "8000",
                "--no-enable-prefix-caching"
            ],
            stdout=archivo,
            stderr=subprocess.STDOUT,
            start_new_session=True
        )

    #Espero a que responda
    while True:
        if servidor.poll() is not None:
            raise RuntimeError("vLLM se ha parado, mirar " + log)
        try:
            requests.get(URL + "/v1/models", timeout=5)
            break
        except Exception:
            time.sleep(10)

    nombre = requests.get(URL + "/v1/models").json()["data"][0]["id"]
    print("vLLM listo con:", nombre)
    return servidor, nombre

def parar(servidor):
    os.killpg(servidor.pid, signal.SIGTERM)
    servidor.wait()
    time.sleep(15)
    print("vLLM parado")

def metricas_spec():
    #Contadores de speculative decoding de vLLM (solo existen con el draft activo)
    valores = {}
    for linea in requests.get(URL + "/metrics").text.splitlines():
        if not linea.startswith("vllm:spec_decode") or "_created" in linea:
            continue
        nombre_completo, valor = linea.rsplit(" ", 1)
        nombre = nombre_completo.split("{")[0]
        posicion = re.search(r'position="(\d+)"', nombre_completo)
        if posicion:
            nombre = nombre + "[" + posicion.group(1) + "]"
        valores[nombre] = valores.get(nombre, 0) + float(valor)
    return valores

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def pedir(modelo, mensajes):
    #Temperatura 0 para que las dos versiones sean comparables
    respuesta = requests.post(URL + "/v1/chat/completions", json={
        "model": modelo,
        "messages": mensajes,
        "temperature": 0,
        "max_tokens": MAX_TOKENS
    }, timeout=900).json()
    return {
        "texto": respuesta["choices"][0]["message"]["content"],
        "tokens": respuesta["usage"]["completion_tokens"]
    }

def evaluar(etiqueta, modelo, concurrencia):
    #Calentamiento para que CUDA graphs y demás no cuenten en la medida
    pedir(modelo, prompts["es"][0])
    pedir(modelo, prompts["ca"][0])

    filas = []
    for idioma, lista in prompts.items():
        antes = metricas_spec()
        inicio = time.time()
        with ThreadPoolExecutor(max_workers=concurrencia) as grupo:
            resultados = list(grupo.map(lambda m: pedir(modelo, m), lista))
        segundos = time.time() - inicio
        despues = metricas_spec()

        diferencia = {k: despues.get(k, 0) - antes.get(k, 0) for k in despues}
        borradores = diferencia.get("vllm:spec_decode_num_drafts_total", 0)
        propuestos = diferencia.get("vllm:spec_decode_num_draft_tokens_total", 0)
        aceptados = diferencia.get("vllm:spec_decode_num_accepted_tokens_total", 0)
        tokens = sum(r["tokens"] for r in resultados)

        fila = {
            "modo": etiqueta,
            "concurrencia": concurrencia,
            "idioma": idioma,
            "tokens": tokens,
            "segundos": round(segundos, 1),
            "tok_s": round(tokens / segundos, 1),
            "longitud_aceptacion": round(1 + aceptados / borradores, 2) if borradores else None,
            "tasa_aceptacion": round(aceptados / propuestos, 3) if propuestos else None,
            "por_posicion": {k.split("[")[1][:-1]: v for k, v in diferencia.items() if "per_pos" in k},
            "textos": [r["texto"] for r in resultados]
        }
        filas.append(fila)
        print(etiqueta, "| c =", concurrencia, "|", idioma, "|", fila["tok_s"], "tok/s | aceptación:", fila["longitud_aceptacion"])
    return filas

In [ ]:
#1) Salamandra sola
servidor, modelo = arrancar("BSC-LT/salamandra-7b-instruct-2606", "/content/vllm_base.log")

resultados = []
resultados += evaluar("base", modelo, 1)
resultados += evaluar("base", modelo, 8)

parar(servidor)

In [ ]:
#2) Salamandra + EAGLE-3, un checkpoint detrás de otro (el checkpoint ya dice qué modelo grande usar)
for etiqueta in CHECKPOINTS:
    servidor, modelo = arrancar("/content/" + etiqueta, "/content/vllm_" + etiqueta + ".log")

    #Compruebo que el speculative decoding está activo de verdad
    !grep -i -m 3 "speculative" /content/vllm_{etiqueta}.log

    resultados += evaluar(etiqueta, modelo, 1)
    resultados += evaluar(etiqueta, modelo, 8)

    parar(servidor)

In [ ]:
import pandas as pd

tabla = pd.DataFrame([{k: v for k, v in fila.items() if k not in ("textos", "por_posicion")} for fila in resultados])

#Speedup = tok/s con EAGLE-3 / tok/s sin aceleración, mismo idioma y misma concurrencia
base = tabla[tabla.modo == "base"].set_index(["concurrencia", "idioma"])["tok_s"]

for etiqueta in CHECKPOINTS:
    eagle = tabla[tabla.modo == etiqueta].set_index(["concurrencia", "idioma"])
    eagle = eagle.assign(tok_s_base=base, speedup=(eagle["tok_s"] / base).round(2))
    print(eagle[["tok_s_base", "tok_s", "speedup", "longitud_aceptacion", "tasa_aceptacion"]])
    print()

In [ ]:
#Aceptación por posición del draft (cuánto acierta el 1º, 2º, 3º token propuesto), con 1 petición a la vez
for fila in resultados:
    if fila["modo"] != "base" and fila["concurrencia"] == 1:
        print(fila["modo"], fila["idioma"], fila["por_posicion"])

In [ ]:
#Comprobación: speculative decoding no debería cambiar las respuestas (a temperatura 0)
#Puede haber alguna diferencia mínima por redondeos numéricos, pero deberían ser casi todas iguales
for etiqueta in CHECKPOINTS:
    for idioma in prompts:
        textos_base = next(f["textos"] for f in resultados if f["modo"] == "base" and f["concurrencia"] == 1 and f["idioma"] == idioma)
        textos_eagle = next(f["textos"] for f in resultados if f["modo"] == etiqueta and f["concurrencia"] == 1 and f["idioma"] == idioma)
        iguales = sum(a == b for a, b in zip(textos_base, textos_eagle))
        print(etiqueta, idioma, "respuestas idénticas:", iguales, "/", len(textos_base))

In [ ]:
#Guardo todo en Drive para el repo y el correo
with open("/content/drive/MyDrive/salamandra-eagle3/eval_checkpoints.json", "w", encoding="utf-8") as archivo:
    json.dump(resultados, archivo, ensure_ascii=False, indent=2)

tabla.to_csv("/content/drive/MyDrive/salamandra-eagle3/eval_checkpoints.csv", index=False)
print("Guardado")